# 직접 수집 1 · 데이터 수집 ① — 시드 & 네트워크 확장

위키피디아를 직접 크롤링해 유망아이템을 발굴하는 과정을 **단계별로** 실행합니다.
전체 흐름: **수집①(네트워크) → 정제(필터) → 수집②(지표) → 결과(PageRank·통계) → 보고서**

## 이 노트북 (수집 ①)
1. **시드 실제 제목 확인** — 입력어를 위키의 정식 문서명으로 정규화
2. **네트워크 확장** — 시드 문서의 **"See also"** 링크를 따라 연관 문서를 모음

> 이후 노트북(02~05)은 이 결과 파일을 이어받습니다. **순서대로 실행**하세요.

In [1]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # wiki_crawling.py 있는 프로젝트 루트 탐색
    if os.path.isfile("wiki_crawling.py"):
        break
    os.chdir("..")
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 2                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

작업 루트: /Users/sumin/Desktop/유망 아이템/wiki_web
대상 문서: Quantum computing | 작업 폴더: runs/Quantum_computing


## 1) 시드 실제 제목 확인
입력한 이름이 리다이렉트/표기 차이가 있을 수 있어, 위키의 **정식 문서명**으로 맞춥니다.

In [2]:
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("입력:", SEED, "→ 실제 제목:", SEED_TRUE)
df_true.head()

[Quantum computing] True Title 확인 (API 호출)...
================ Quantum computing =============================
[0, 'Quantum computing', 'Quantum computing', '', 'A quantum computer is a real or theoretical computer that exploits quantum phenomena like superposition and entanglement in an essential way. It is widely believed that a quantum computer could perform some calculations exponentially faster than any classical computer. For example, a large-scale quantum computer could break some widely used encryption schemes and aid physicists in performing physical simulations. However, current hardware implementations of quantum computation are largely experimental and only suitable for specialized tasks. The basic unit of information in quantum computing, the qubit (or "quantum bit"), serves the same function as the bit in ordinary or "classical" computing. However, unlike a classical bit, which can be in one of two states (a binary), a qubit can exist in a linear combination of two states

,No,Title,True_title,Category,Contents,URL
0,0,Quantum computing,Quantum computing,,A quantum computer is a real or theoretical co...,https://en.wikipedia.org/wiki/Quantum_computing


## 2) 네트워크 확장의 원리 — "See also"
확장은 문서 위키텍스트의 **`== See also ==`** 섹션 링크를 따라갑니다.
(일반 `[[링크]]` + `{{Annotated link|...}}` 템플릿 모두 인식하도록 파서를 보강했습니다.)

먼저 시드 문서의 See also 링크를 직접 뽑아봅니다.

In [3]:
sess = wc._create_session()
r = sess.get("https://en.wikipedia.org/w/api.php", params={
    "action": "query", "format": "json", "prop": "revisions",
    "rvprop": "content", "rvslots": "main", "titles": SEED_TRUE, "formatversion": "2"
}, verify=False, timeout=25)
wikitext = r.json()["query"]["pages"][0]["revisions"][0]["slots"]["main"]["content"]

see_also = wc._parse_see_also(wikitext)
print(f"'{SEED_TRUE}'의 See also 링크: {len(see_also)}개")
see_also

'Quantum computing'의 See also 링크: 26개


['List of quantum computing journals',
 'List of computer books',
 'List of quantum software',
 'D-Wave Systems',
 'Electronic quantum holography',
 'Glossary of quantum computing',
 'Intelligence Advanced Research Projects Activity',
 "India's quantum computer",
 'QpiAI-Indus',
 'IonQ',
 'List of emerging technologies',
 'Magic state distillation',
 'Metacomputing',
 'Natural computing',
 'Non-local quantum computation',
 'Optical computing',
 'Quantum bus',
 'Quantum cognition',
 'Quantum sensor',
 'Quantum volume',
 'Quantum weirdness',
 'Rigetti Computing',
 'Supercomputer',
 'Theoretical computer science',
 'Unconventional computing',
 'Valleytronics']

## 3) 전체 네트워크 확장 실행 (1차수)
`expand_network`가 시드의 See also 문서들을 크롤링해 **엣지(시드 → 연관문서)** 표를 만듭니다.

In [4]:
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
edf = pd.read_excel(EXPAND)
nodes = set(edf["From_title"].dropna()) | set(edf["To_seealso"].dropna())
print(f"확장 엣지 {len(edf)}개 · 노드 {len(nodes)}개  →  {EXPAND}")
edf.head(15)

[Quantum computing] 위키 네트워크 확장 중 (n=2)...
0 ['Quantum computing']
1차시 시작 (frontier=1)
1차시입니다. 수집 26건, 다음 frontier 26건
2차시 시작 (frontier=26)
2차시입니다. 수집 158건, 다음 frontier 143건
확장 엣지 184개 · 노드 170개  →  runs/Quantum_computing/seed_item/Quantum computing/2차시 확장 최종_결과.xlsx


,From_title,Fake_seealso,To_seealso,Category,Contents
0,Quantum computing,List of quantum computing journals,List of quantum computing journals,Computer science journals| Lists of academic j...,NaN
1,Quantum computing,List of computer books,List of computer books,Books about computer hacking| Computer books| ...,NaN
2,Quantum computing,List of quantum software,List of quantum software,Computing-related lists| Quantum computing| Qu...,NaN
3,Quantum computing,D-Wave Systems,D-Wave Systems,1999 establishments in British Columbia| Compa...,NaN
4,Quantum computing,Electronic quantum holography,Electronic quantum holography,Holographic data storage,NaN
5,Quantum computing,Glossary of quantum computing,Glossary of quantum computing,Classes of computers| Computational complexity...,NaN
6,Quantum computing,Intelligence Advanced Research Projects Activity,Intelligence Advanced Research Projects Activity,2006 establishments in the United States| Gove...,NaN
7,Quantum computing,India's quantum computer,India's quantum computer,Quantum computing| Science and technology in I...,NaN
8,Quantum computing,QpiAI-Indus,QpiAI-Indus,Quantum computing| Science and technology in I...,NaN
9,Quantum computing,IonQ,IonQ,2021 initial public offerings| Companies invol...,NaN


**정리**
- 시드 → See also 연관 문서로 **네트워크(엣지)** 를 확보했습니다.
- 다음 노트북(**02 정제**)에서 목록성 문서 등 노이즈를 걸러냅니다.